In [ ]:
%%capture
!pip install openai-whisper google-cloud-storage pandas tqdm

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import whisper
import pandas as pd
import json
import os
from google.cloud import storage
from pathlib import Path
import time
from tqdm import tqdm
import tempfile
import numpy as np
from io import StringIO

In [ ]:
# full dataset transcription with temp file reading
bucket_name = "taukadial-25"
audio_folder = "audio-data"
groundtruth_path = "groundtruth/groundtruth_combined_lang.csv"
output_csv = "transcription_data.csv"
model_size = "base"

# Initialize GCS client
client = storage.Client()
bucket = client.bucket(bucket_name)

# Load groundtruth data
blob = bucket.blob(groundtruth_path)
csv_content = blob.download_as_text()
df = pd.read_csv(StringIO(csv_content))

# Load Whisper model
model = whisper.load_model(model_size)

# prep results lists
results = []
failed_files = []

print(f"Processing {len(df)} audio files...")

# Process each file
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Transcribing files"):
    filename = row['tkdname']
    label = row['dx']

    temp_file_path = None

    # Get blob from GCS
    blob_path = f"{audio_folder}/{filename}"
    blob = bucket.blob(blob_path)

    temp_audio_file = tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(filename)[1] or ".wav")
    blob.download_to_file(temp_audio_file)
    temp_file_path = temp_audio_file.name
    temp_audio_file.close()

    result = model.transcribe(temp_file_path)

    # Store results
    results.append({
        'tkdname': filename,
        'dx': label,
        'detected_language': result['language'],
        'transcription': result['text'].strip()
    })

    # cleanup
    if temp_file_path and os.path.exists(temp_file_path):
      os.remove(temp_file_path)

# Create results DataFrame
results_df = pd.DataFrame(results)

# Save to CSV
results_df.to_csv(output_csv, index=False)
print(f"✅ Saved {len(results_df)} transcriptions to {output_csv}")

# Print summary statistics
print(f"\n📊 SUMMARY:")
print(f"Total files processed: {len(df)}")
print(f"Successful transcriptions: {len(results_df)}")
print(f"Failed transcriptions: {len(failed_files)}")

print(f"\nLanguage distribution:")
print(results_df['detected_language'].value_counts())

print(f"\nLabel distribution:")
print(results_df['dx'].value_counts())

print(f"\nAverage transcription length: {results_df['transcription'].str.len().mean():.1f} characters")

# Save failed files log if any
if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv("failed_transcriptions.csv", index=False)
    print(f"⚠️  Saved {len(failed_files)} failed files to failed_transcriptions.csv")

print("\n✅ Transcription pipeline completed!")

100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 54.2MiB/s]



3. Processing 507 audio files...


Transcribing files: 100%|██████████| 507/507 [33:46<00:00,  4.00s/it]

✅ Saved 507 transcriptions to transcription_data.csv

📊 SUMMARY:
Total files processed: 507
Successful transcriptions: 507
Failed transcriptions: 0

Language distribution:
detected_language
zh    255
en    246
my      5
bo      1
Name: count, dtype: int64

Label distribution:
dx
MCI    285
NC     222
Name: count, dtype: int64

Average transcription length: 424.1 characters

✅ Transcription pipeline completed!


In [ ]:
# Upload the transcription_data.csv to GCS
output_blob_name = f"groundtruth/{output_csv}" # or any other desired path in your bucket
blob = bucket.blob(output_blob_name)
blob.upload_from_filename(output_csv)

print(f"✅ Uploaded {output_csv} to gs://{bucket_name}/{output_blob_name}")

✅ Uploaded transcription_data.csv to gs://taukadial-25/groundtruth/transcription_data.csv


In [ ]:
results_df.head()

,tkdname,dx,detected_language,transcription
0,taukdial-002-1.wav,NC,en,Yes. Do you need to zoom in or anything? No. O...
1,taukdial-002-2.wav,NC,en,"Okay. With the beginning, middle and an end. O..."
2,taukdial-002-3.wav,NC,en,"Yes. Okay, can you just tell me everything tha..."
3,taukdial-003-1.wav,MCI,zh,"這個小朋友在抓魚用老魚,老魚,老魚,西西魚,我在玩水"
4,taukdial-003-2.wav,MCI,zh,"這個打平坊球就走上去這是國就是國要咬他,不要做什麼呢而且很懂得"


##post transcription processing

In [ ]:
# print filename and detected language for rows that aren't en / zh
non_en_zh_df = results_df[~results_df['detected_language'].isin(['en', 'zh'])]

print("Files with non-English or non-Chinese detected language:")
if not non_en_zh_df.empty:
    display(non_en_zh_df[['tkdname', 'detected_language']])
else:
    print("No files found with non-English or non-Chinese detected language.")

Files with non-English or non-Chinese detected language:


,tkdname,detected_language
6,taukdial-004-1.wav,my
7,taukdial-004-2.wav,my
8,taukdial-004-3.wav,my
216,taukdial-100-1.wav,bo
273,taukdial-122-1.wav,my
368,taukdial-161-3.wav,my


In [ ]:
# Create a copy of the results_df
results_df_modified = results_df.copy()

# Identify rows where the detected language is not 'en' or 'zh'
mask_non_en_zh = ~results_df_modified['detected_language'].isin(['en', 'zh'])

# Replace the detected_language with 'zh' for these rows
results_df_modified.loc[mask_non_en_zh, 'detected_language'] = 'zh'

# Display the updated language distribution to verify
print("Language distribution in results_df_modified:")
display(results_df_modified['detected_language'].value_counts())

results_df_modified.to_csv("transcription_data_modified.csv", index=False)

Language distribution in results_df_modified:


,count
detected_language,
zh,261
en,246


## single file transcription test

In [ ]:
# single file transcription test
bucket_name = "taukadial-25"
audio_folder = "audio-data"
model_size = "base"

client = storage.Client()
bucket = client.bucket(bucket_name)

# load groundtruth
blob = bucket.blob("groundtruth/groundtruth_combined_lang.csv")
csv_content = blob.download_as_text()
df = pd.read_csv(StringIO(csv_content))

test_filename = df['tkdname'].iloc[10]

# Download to a temporary file
blob_path = f"{audio_folder}/{test_filename}"
blob = bucket.blob(blob_path)

temp_audio_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") # Use appropriate suffix based on your audio files
blob.download_to_file(temp_audio_file)
temp_file_path = temp_audio_file.name
temp_audio_file.close() # Close the file handle after writing
print(f"✅ Download successful - Audio saved to temporary file: '{temp_file_path}'")

# whisper
model = whisper.load_model(model_size)
audio_np = whisper.load_audio(temp_file_path)
result = model.transcribe(audio_np)

print("✅ Transcription successful!")
print(f"Detected language: {result['language']}")
print(f"Transcription length: {len(result['text'])} characters")
print(f"Transcription: {result['text']}")

# Show segments if available
if 'segments' in result and len(result['segments']) > 0:
    print(f"Number of segments: {len(result['segments'])}")
    print(f"First segment: {result['segments'][0]}")

if os.path.exists(temp_file_path):
    os.remove(temp_file_path)
    print("✅ Temporary file cleaned up.")

✅ Download successful - Audio saved to temporary file: '/tmp/tmp6od7emye.mp3'
✅ Transcription successful!
Detected language: en
Transcription length: 396 characters
Transcription:  Okay, looks like a little kitty cat got trapped in the tree and the little girls franny. So it looks like her father came over to Or a man came over to help get the kitty cat out of the tree in the meantime, the dogs barking. And the fire department has arrived to help with A ladder to help rescue the cat. Yeah. And hopefully this will all resolve with a kitty cat being rescued from the tree.
Number of segments: 6
First segment: {'id': 0, 'seek': 0, 'start': 0.0, 'end': 9.56, 'text': ' Okay, looks like a little kitty cat got trapped in the tree and the little girls franny. So it looks like her father came over to', 'tokens': [50364, 1033, 11, 1542, 411, 257, 707, 33026, 3857, 658, 14994, 294, 264, 4230, 293, 264, 707, 4519, 431, 11612, 13, 407, 309, 1542, 411, 720, 3086, 1361, 670, 281, 50842], 'temperature'